# Burn severity and erosion priority after the Vesuvius fire of August 2025

On the evening of 8 August 2025 a fire started above Terzigno, on the south-east flank of Vesuvius, and burned for four days inside the national park. Copernicus Emergency Management Service opened activation EMSR830 for it on 9 August. The press reported about 500 hectares of pine and shrub lost. This notebook measures how badly each pixel burned, from Sentinel-2 images taken before and after the fire, and then asks one practical question: which burned slopes should the park authority and the Campania civil protection treat first, before the autumn rains wash the bare ash off the steep ground?

The answer comes out as three things a non-EO reader can use: a ranked table of 250 m cells, a GeoJSON of the flagged cells, and a short plain-language summary. Every pixel is downloaded from Microsoft Planetary Computer (Sentinel-2 L2A and the Copernicus DEM) and processed here; nothing is computed on a remote server, and no account or API key is needed. To run it: Python 3.12, `pip install -r requirements.txt`, `pip install -e .`, then run all cells. The README has the Colab variant.

## 1. Where, when and the rules

The study box is 14.35 to 14.50 east and 40.77 to 40.87 north, about 13 by 11 km around the cone and its south-east flank, where the fire burned. Two time windows feed the analysis. The pre-fire window runs from 1 June to 7 August 2025 and the post-fire window from 13 August to 15 October 2025; every clear scene in a window goes into a per-pixel median, so one missed cloud does not spoil the picture. Severity comes from dNBR in the five classes of Key and Benson (2006). The decision rule follows the terrain term of the USGS post-fire debris-flow model M1 (Staley et al. 2017): burned at moderate or high severity, which starts at dNBR 0.27 in those classes, on a slope of 23 degrees or more. Only that terrain term is computed; the full model also needs rainfall intensity and soil erodibility. All these numbers live in `src/burnsev/aoi.py`, each with its reason, so anyone can change one and rerun.

In [1]:
# Step 1. Every number a user might want to change lives in src/burnsev/aoi.py, with its reason next to it.
from burnsev import aoi

print("AOI (west, south, east, north):", aoi.BBOX)
print("Fire:", aoi.FIRE_START, "to", aoi.FIRE_END)
print("Pre-fire window: ", *aoi.PRE_WINDOW)
print("Post-fire window:", *aoi.POST_WINDOW)
print("Bands:", aoi.BANDS, "at", aoi.RESOLUTION, "m in", aoi.CRS)
print("Severe = dNBR >=", aoi.SEVERITY_CUT_DNBR, "->", aoi.SEVERE_CLASSES)
print("Steep  = slope >=", aoi.SLOPE_THRESHOLD_DEG, "deg; sensitivity at", aoi.SLOPE_SENSITIVITY_DEG)

AOI (west, south, east, north): (14.35, 40.77, 14.5, 40.87)
Fire: 2025-08-08 to 2025-08-12
Pre-fire window:  2025-06-01 2025-08-07
Post-fire window: 2025-08-13 2025-10-15
Bands: ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12', 'SCL'] at 20 m in EPSG:32633
Severe = dNBR >= 0.27 -> ('moderate-low', 'moderate-high', 'high')
Steep  = slope >= 23.0 deg; sensitivity at (20.0, 23.0, 26.0)


## 2. Which scenes exist

The catalogue search returns 31 scenes in the pre-fire window and 23 after the fire, all with less than 25 percent cloud over the whole tile. They come from three satellites (Sentinel-2A, 2B and the newer 2C) on two orbits, 79 and 122, which see the box from slightly different angles a few days apart; that is why some dates appear twice. The first post-fire scene is 14 August, two days after containment. Every scene is processing baseline 05.11, so the table shows the same offset of minus 1000 for all of them, read from the metadata of each scene rather than assumed. One acquisition on 7 June was published twice, and only the newer product is kept, otherwise that date would count twice in the median. The cloud filter is applied to the returned metadata in Python, because the query extension is optional in STAC and this catalogue does not advertise it.

In [2]:
# Step 2. Ask the catalogue which scenes exist over the box in each window. Metadata only, no pixels yet.
from burnsev import catalog

pre_items = catalog.search_scenes(aoi.BBOX, *aoi.PRE_WINDOW, aoi.MAX_CLOUD)
post_items = catalog.search_scenes(aoi.BBOX, *aoi.POST_WINDOW, aoi.MAX_CLOUD)
print(len(pre_items), "pre-fire and", len(post_items), "post-fire scenes under", aoi.MAX_CLOUD, "% cloud")

scenes = catalog.scene_table(pre_items + post_items)
scenes

31 pre-fire and 23 post-fire scenes under 25.0 % cloud


,id,date,satellite,orbit,tile,cloud_pct,baseline,boa_offset
0,S2C_MSIL2A_20250605T095051_R079_T33TVF_2025060...,2025-06-05,Sentinel-2C,79,33TVF,1.8,05.11,-1000
1,S2A_MSIL2A_20250607T095041_R079_T33TVF_2025060...,2025-06-07,Sentinel-2A,79,33TVF,0.0,05.11,-1000
2,S2C_MSIL2A_20250608T100051_R122_T33TVF_2025060...,2025-06-08,Sentinel-2C,122,33TVF,9.5,05.11,-1000
3,S2B_MSIL2A_20250610T095029_R079_T33TVF_2025061...,2025-06-10,Sentinel-2B,79,33TVF,2.5,05.11,-1000
4,S2A_MSIL2A_20250610T100041_R122_T33TVF_2025061...,2025-06-10,Sentinel-2A,122,33TVF,2.4,05.11,-1000
5,S2B_MSIL2A_20250613T100029_R122_T33TVF_2025061...,2025-06-13,Sentinel-2B,122,33TVF,0.2,05.11,-1000
6,S2C_MSIL2A_20250615T095051_R079_T33TVF_2025061...,2025-06-15,Sentinel-2C,79,33TVF,0.6,05.11,-1000
7,S2A_MSIL2A_20250617T095051_R079_T33TVF_2025061...,2025-06-17,Sentinel-2A,79,33TVF,12.6,05.11,-1000
8,S2B_MSIL2A_20250620T095029_R079_T33TVF_2025062...,2025-06-20,Sentinel-2B,79,33TVF,1.5,05.11,-1000
9,S2A_MSIL2A_20250620T100041_R122_T33TVF_2025062...,2025-06-20,Sentinel-2A,122,33TVF,1.0,05.11,-1000


## 3. Load the pixels and mask the clouds

> _Narration drafted when this step is built._ odc-stac loads the cube; SCL mask; DN to reflectance with the offset; clear-pixel share per date.

In [3]:
# Step 3: written after step 2 is approved.

## 4. Look before and after

> _Narration drafted when this step is built._ True colour, pre and post, same 2 to 98 percent stretch. One figure.

In [4]:
# Step 4: written after step 3 is approved.

## 5. Composites, NBR and dNBR

> _Narration drafted when this step is built._ Median composites per window, NBR of each, dNBR = pre minus post. One figure.

In [5]:
# Step 5: written after step 4 is approved.

## 6. Severity classes and areas

> _Narration drafted when this step is built._ Key and Benson classes; map and a table of hectares per class.

In [6]:
# Step 6: written after step 5 is approved.

## 7. Check against the EFFIS perimeter

> _Narration drafted when this step is built._ Independent reference, not ground truth: area, intersection over union, where they disagree.

In [7]:
# Step 7: written after step 6 is approved.

## 8. Terrain: DEM and slope

> _Narration drafted when this step is built._ Copernicus DEM GLO-30 from the same catalogue, reprojected to the cube grid; slope in degrees.

In [8]:
# Step 8: written after step 7 is approved.

## 9. The decision: which slopes first

> _Narration drafted when this step is built._ 250 m grid, score per cell, ranked table, alert GeoJSON, plain summary; sensitivity at 20, 23 and 26 degrees.

In [9]:
# Step 9: written after step 8 is approved.

## 10. Files written

> _Narration drafted when this step is built._ COG, GeoJSON, PNGs; list with sizes; COG validated.

In [10]:
# Step 10: written after step 9 is approved.

## 11. Limitations and next steps

> _Narration drafted when this step is built._ Short list: what the numbers cannot say, what was cut and why, what comes next.

## Bonus A. The pipeline as an MCP tool, called by an LLM

> _Narration drafted when this step is built._ After the core is green.

In [11]:
# Bonus A: after the core is green.

## Bonus B. A geospatial foundation model on the same fire

> _Narration drafted when this step is built._ After bonus A.

In [12]:
# Bonus B: after bonus A.